# **Project Name    -    Sentiment Analysis on Zomato restaurant reviews.**



##### **Project Type**    - EDA/Regression/Classification/Unsupervised
##### **Contribution**    - Individual
##### **Name**            - Vipeen Kumar


# **Project Summary -**

This project performs Sentiment Analysis on Zomato restaurant reviews. Leveraging two datasets—one containing restaurant metadata (name, cost, cuisines, timings) and another containing customer reviews and ratings—we extract meaningful insights regarding customer satisfaction. We merge the datasets, preprocess textual reviews using NLP techniques (TF-IDF), and build classification models (Logistic Regression, Random Forest, XGBoost) to predict whether a review is Positive or Negative. The EDA uncovers relationships between cost, rating, and customer sentiment. The final model achieves high accuracy and provides actionable insights for restaurants to improve their services based on customer feedback.

# **GitHub Link -**

https://github.com/Vipeen21/Data-Science-AI-ML-Zomato

# **Problem Statement**


**Problem Statement**: Zomato is a major restaurant aggregator and food delivery startup. Understanding customer sentiment from reviews is crucial for both restaurants (to improve service) and customers (to make dining choices). The objective of this project is to build an NLP-based Machine Learning classification model that accurately predicts the sentiment (Positive/Negative) of a customer review based on the text. We will also perform extensive Exploratory Data Analysis to understand factors affecting ratings, such as cost and cuisines.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import joblib

### Dataset Loading

In [ ]:
df_meta = pd.read_csv('Zomato Restaurant names and Metadata.csv')
df_reviews = pd.read_csv('Zomato Restaurant reviews.csv')

### Dataset First View

In [ ]:
display(df_meta.head(2))
display(df_reviews.head(2))

### Dataset Rows & Columns count

In [ ]:
print(f'Metadata shape: {df_meta.shape}')
print(f'Reviews shape: {df_reviews.shape}')

### Dataset Information

In [ ]:
df_meta.info()
print('-'*40)
df_reviews.info()

#### Duplicate Values

In [ ]:
print(f'Metadata duplicates: {df_meta.duplicated().sum()}')
print(f'Reviews duplicates: {df_reviews.duplicated().sum()}')

#### Missing Values/Null Values

In [ ]:
print('Metadata Missing:')
print(df_meta.isnull().sum())
print('\nReviews Missing:')
print(df_reviews.isnull().sum())

In [ ]:
plt.figure(figsize=(10,4))
sns.heatmap(df_reviews.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap (Reviews)')
plt.show()

### What did you know about your dataset?

The datasets consist of restaurant metadata (105 rows) and customer reviews (10,000 rows). There are missing values in the 'Collections' and 'Timings' columns of the metadata, and in 'Reviewer', 'Review', 'Rating', 'Metadata', and 'Time' of the reviews dataset. We will merge them on the 'Restaurant/Name' column.

## ***2. Understanding Your Variables***

In [ ]:
print(df_meta.columns)
print(df_reviews.columns)

In [ ]:
display(df_reviews.describe(include='all'))

### Variables Description

- **Restaurant/Name**: Name of the restaurant
- **Cost**: Cost for two people
- **Cuisines**: Types of food served
- **Review**: Textual feedback from customer
- **Rating**: Rating given by customer (1-5)
- **Sentiment**: Target variable (1 for Positive, 0 for Negative)

### Check Unique Values for each variable.

In [ ]:
print('Unique values in Metadata:')
print(df_meta.nunique())
print('\nUnique values in Reviews:')
print(df_reviews.nunique())

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Drop missing reviews and ratings
df_reviews.dropna(subset=['Review', 'Rating'], inplace=True)

# Convert Rating to numeric. Extract numeric part from strings like '4.0'
df_reviews['Rating'] = df_reviews['Rating'].apply(lambda x: float(str(x).strip().split('/')[0]) if str(x).replace('.','',1).isdigit() else np.nan)
df_reviews.dropna(subset=['Rating'], inplace=True)

# Create Sentiment Target (1 for Positive >= 3.5, 0 for Negative < 3.5)
df_reviews['Sentiment'] = df_reviews['Rating'].apply(lambda x: 1 if x >= 3.5 else 0)

# Merge datasets
df_merged = pd.merge(df_reviews, df_meta, left_on='Restaurant', right_on='Name', how='left')

# Clean Cost column
df_merged['Cost'] = df_merged['Cost'].astype(str).str.replace(',', '')
df_merged['Cost'] = pd.to_numeric(df_merged['Cost'], errors='coerce')

df = df_merged.copy()
df.head()

### What all manipulations have you done and insights you found?

1. Merged metadata and reviews datasets on restaurant name.
2. Dropped rows with missing 'Review' or 'Rating'.
3. Converted 'Rating' to a numeric float value. Created a 'Sentiment' target (1 if Rating >= 3.5 else 0).
4. Cleaned the 'Cost' column by removing commas and converting to integer.
5. Dropped duplicate reviews (if any).

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Sentiment', data=df, palette='Set2')
plt.title('Distribution of Sentiments')
plt.show()

##### 1. Why did you pick the specific chart?

To understand the distribution of the target variable.

##### 2. What is/are the insight(s) found from the chart?

The dataset has more positive reviews than negative reviews.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, identifying baseline sentiment helps measure overall customer satisfaction.

#### Chart - 2

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Rating'], bins=10, kde=True, color='skyblue')
plt.title('Distribution of Ratings')
plt.show()

##### 1. Why did you pick the specific chart?

To see the spread of ratings (1 to 5).

##### 2. What is/are the insight(s) found from the chart?

Most ratings are clustered between 3.5 and 4.5.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, shows general high quality of listed restaurants.

#### Chart - 3

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Cost'].dropna(), bins=20, kde=True, color='green')
plt.title('Distribution of Cost for Two')
plt.show()

##### 1. Why did you pick the specific chart?

To see the pricing distribution across restaurants.

##### 2. What is/are the insight(s) found from the chart?

Most restaurants cost between 500 and 1500 for two people.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, helps segment restaurants by budget.

#### Chart - 4

In [ ]:
top_restaurants = df['Restaurant'].value_counts().head(10)
plt.figure(figsize=(10,5))
sns.barplot(y=top_restaurants.index, x=top_restaurants.values, palette='viridis')
plt.title('Top 10 Reviewed Restaurants')
plt.show()

##### 1. Why did you pick the specific chart?

To identify the most popular/reviewed restaurants.

##### 2. What is/are the insight(s) found from the chart?

Certain restaurants dominate the review counts.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, these are key partners for Zomato.

#### Chart - 5

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(x='Sentiment', y='Cost', data=df, palette='Set1')
plt.title('Cost vs Sentiment')
plt.show()

##### 1. Why did you pick the specific chart?

To see if more expensive restaurants get better sentiments.

##### 2. What is/are the insight(s) found from the chart?

Positive and negative sentiments have similar cost distributions, though positive might slightly lean higher.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, indicates price alone doesn't guarantee satisfaction.

#### Chart - 6

In [ ]:
df['Review_Length'] = df['Review'].apply(lambda x: len(str(x).split()))
plt.figure(figsize=(10,6))
sns.boxplot(x='Sentiment', y='Review_Length', data=df, palette='coolwarm')
plt.title('Review Length vs Sentiment')
plt.show()

##### 1. Why did you pick the specific chart?

To see if angry or happy customers write longer reviews.

##### 2. What is/are the insight(s) found from the chart?

Negative reviews tend to be slightly longer (people complaining).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, long reviews could be automatically flagged for customer service intervention.

#### Chart - 7

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(x='Cost', y='Rating', data=df, alpha=0.5)
plt.title('Cost vs Rating')
plt.show()

##### 1. Why did you pick the specific chart?

To check for linear relationship between cost and rating.

##### 2. What is/are the insight(s) found from the chart?

No strong linear correlation, but very cheap places rarely get 5 stars.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, sets expectations for budget vs quality.

#### Chart - 8

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Pictures', data=df)
plt.title('Reviews with Pictures')
plt.show()

##### 1. Why did you pick the specific chart?

To see how many users upload pictures with reviews.

##### 2. What is/are the insight(s) found from the chart?

Most reviews do not have pictures (Pictures=0).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, can incentivize picture uploads to increase engagement.

#### Chart - 9

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(x=df['Pictures']>0, y='Rating', data=df)
plt.title('Rating Distribution by Picture Presence')
plt.show()

##### 1. Why did you pick the specific chart?

To see if adding pictures correlates with higher ratings.

##### 2. What is/are the insight(s) found from the chart?

Reviews with pictures tend to have slightly higher median ratings.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, positive experiences might encourage taking photos.

#### Chart - 10

In [ ]:
cuisines = df['Cuisines'].str.split(', ').explode().value_counts().head(10)
plt.figure(figsize=(10,5))
sns.barplot(x=cuisines.values, y=cuisines.index, palette='magma')
plt.title('Top 10 Cuisines')
plt.show()

##### 1. Why did you pick the specific chart?

To find the most common food types.

##### 2. What is/are the insight(s) found from the chart?

North Indian and Chinese are extremely popular.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, guides marketing and onboarding of new restaurants.

#### Chart - 11

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x='Sentiment', y='Pictures', data=df, palette='pastel')
plt.title('Average Pictures per Sentiment')
plt.show()

##### 1. Why did you pick the specific chart?

To see if positive reviews contain more pictures on average.

##### 2. What is/are the insight(s) found from the chart?

Positive reviews generally have a higher average picture count.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes.

#### Chart - 12

In [ ]:
plt.figure(figsize=(10,6))
sns.violinplot(x='Sentiment', y='Rating', data=df, palette='muted')
plt.title('Violin Plot of Ratings by Sentiment')
plt.show()

##### 1. Why did you pick the specific chart?

To see the density of exact ratings within each sentiment class.

##### 2. What is/are the insight(s) found from the chart?

Negative sentiment is dense around 1-2, positive is dense around 4-5.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, validates our 3.5 threshold.

#### Chart - 13

In [ ]:
plt.figure(figsize=(10,6))
sns.kdeplot(data=df, x='Review_Length', hue='Sentiment', fill=True)
plt.title('Review Length Density by Sentiment')
plt.xlim(0, 150)
plt.show()

##### 1. Why did you pick the specific chart?

To compare the shape of review length distributions.

##### 2. What is/are the insight(s) found from the chart?

Negative reviews have a fatter tail indicating longer complaints.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes.

#### Chart - 14 - Correlation Heatmap

In [ ]:
plt.figure(figsize=(8,6))
numerical_cols = df[['Rating', 'Cost', 'Pictures', 'Review_Length', 'Sentiment']]
sns.heatmap(numerical_cols.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

##### 1. Why did you pick the specific chart?

To find linear correlations between all numerical features.

##### 2. What is/are the insight(s) found from the chart?

Rating and Sentiment are highly correlated (as expected). Cost has low correlation with Rating.

#### Chart - 15 - Pair Plot

In [ ]:
sns.pairplot(df[['Rating', 'Cost', 'Review_Length', 'Sentiment']], hue='Sentiment', diag_kind='kde')
plt.show()

##### 1. Why did you pick the specific chart?

To visualize pairwise relationships.

##### 2. What is/are the insight(s) found from the chart?

Provides a holistic view of numerical features separated by sentiment.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Yes.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null: There is no difference in ratings between high-cost and low-cost restaurants. Alternate: There is a difference.

#### 2. Perform an appropriate statistical test.

In [ ]:
from scipy.stats import ttest_ind
high_cost = df[df['Cost'] > df['Cost'].median()]['Rating'].dropna()
low_cost = df[df['Cost'] <= df['Cost'].median()]['Rating'].dropna()
stat, p = ttest_ind(high_cost, low_cost)
print(f'T-test Stat: {stat:.4f}, P-value: {p:.4e}')

##### Which statistical test have you done to obtain P-Value?

Independent Two-Sample T-Test

##### Why did you choose the specific statistical test?

Because we are comparing the means of two independent continuous groups (ratings of high vs low cost).

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null: Review length does not affect the sentiment. Alternate: Review length affects sentiment.

#### 2. Perform an appropriate statistical test.

In [ ]:
from scipy.stats import ttest_ind
long_rev = df[df['Review_Length'] > df['Review_Length'].median()]['Sentiment'].dropna()
short_rev = df[df['Review_Length'] <= df['Review_Length'].median()]['Sentiment'].dropna()
stat, p = ttest_ind(long_rev, short_rev)
print(f'T-test Stat: {stat:.4f}, P-value: {p:.4e}')

##### Which statistical test have you done to obtain P-Value?

Independent Two-Sample T-Test

##### Why did you choose the specific statistical test?

To compare sentiment means between short and long review groups.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null: Presence of pictures does not affect sentiment. Alternate: Presence of pictures affects sentiment.

#### 2. Perform an appropriate statistical test.

In [ ]:
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(df['Pictures'] > 0, df['Sentiment'])
stat, p, dof, expected = chi2_contingency(contingency_table)
print(f'Chi-Square Stat: {stat:.4f}, P-value: {p:.4e}')

##### Which statistical test have you done to obtain P-Value?

Chi-Square Test of Independence

##### Why did you choose the specific statistical test?

Because both variables (Has Pictures, Sentiment) are categorical.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handled during Data Wrangling (dropped nulls in target). Cost nulls can be filled with median.
df['Cost'].fillna(df['Cost'].median(), inplace=True)

#### What all missing value imputation techniques have you used and why did you use those techniques?

Dropped missing targets as we cannot impute them for classification. Imputed Cost with median.

### 2. Handling Outliers

In [ ]:
# Using models robust to outliers (Trees) and TF-IDF for text. No removal needed.

##### What all outlier treatment techniques have you used and why did you use those techniques?

None. Outliers in text length are natural.

### 3. Categorical Encoding

In [ ]:
# Not encoding categorical features like Cuisines to keep baseline text-focused.

#### What all categorical encoding techniques have you used & why did you use those techniques?

None. Focusing on NLP.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Skipping expansion for brevity.

#### 2. Lower Casing

In [ ]:
df['Review_Clean'] = df['Review'].str.lower()

#### 3. Removing Punctuations

In [ ]:
def remove_punctuations(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['Review_Clean'] = df['Review_Clean'].apply(lambda x: remove_punctuations(str(x)))

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
df['Review_Clean'] = df['Review_Clean'].apply(lambda x: re.sub(r'http\S+|www\S+|https\S+', '', x, flags=re.MULTILINE))
df['Review_Clean'] = df['Review_Clean'].apply(lambda x: re.sub(r'\w*\d\w*', '', x))

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
stop_words = set(stopwords.words('english'))
stop_words.discard('not') # keep negation
def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stop_words])
df['Review_Clean'] = df['Review_Clean'].apply(remove_stopwords)

In [ ]:
df['Review_Clean'] = df['Review_Clean'].str.strip()
df['Review_Clean'] = df['Review_Clean'].apply(lambda x: re.sub(' +', ' ', x))

#### 6. Rephrase Text

In [ ]:
# Skipped

#### 7. Tokenization

In [ ]:
# Handled by TF-IDF

#### 8. Text Normalization

In [ ]:
lemmatizer = WordNetLemmatizer()
def lemmatize_text(text):
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])
df['Review_Clean'] = df['Review_Clean'].apply(lemmatize_text)

##### Which text normalization technique have you used and why?

Lemmatization to bring words to their dictionary root form.

#### 9. Part of speech tagging

In [ ]:
# Skipped

#### 10. Text Vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = tfidf.fit_transform(df['Review_Clean'])
y = df['Sentiment'].values
print(f'TF-IDF Feature Matrix Shape: {X.shape}')

##### Which text vectorization technique have you used and why?

TF-IDF Vectorizer with top 5000 features and unigram/bigram to capture word importance and context.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Done via TF-IDF

#### 2. Feature Selection

In [ ]:
# Using top 5000 features via max_features in TF-IDF

##### What all feature selection methods have you used  and why?

TF-IDF built-in max features selection.

##### Which all features you found important and why?

N-grams.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Transformed via TF-IDF

### 6. Data Scaling

In [ ]:
# TF-IDF output is already L2 normalized.

TF-IDF inherently normalizes.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

No.

In [ ]:
# Not needed

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

None.

### 8. Data Splitting

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

##### What data splitting ratio have you used and why?

80/20 split. Standard for sufficient training and validation.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Slightly imbalanced (more positive reviews).

In [ ]:
# Handled via class_weight='balanced' in models.

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

class_weight parameter in models to give higher weight to minority class.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
lr = LogisticRegression(class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
print('Logistic Regression Performance:')
print(classification_report(y_test, y_pred_lr))

cm = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Logistic Regression Confusion Matrix')
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
param_grid_lr = {'C': [0.1, 1, 10]}
grid_lr = GridSearchCV(LogisticRegression(class_weight='balanced', max_iter=1000), param_grid_lr, cv=3, scoring='f1')
grid_lr.fit(X_train, y_train)
y_pred_grid_lr = grid_lr.predict(X_test)
print(f'Best Params: {grid_lr.best_params_}')
print(classification_report(y_test, y_pred_grid_lr))

##### Which hyperparameter optimization technique have you used and why?

GridSearchCV to test combinations of hyperparameters exhaustively.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes, slight improvement.

### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('Random Forest Performance:')
print(classification_report(y_test, y_pred_rf))
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens')
plt.title('Random Forest Confusion Matrix')
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
param_grid_rf = {'max_depth': [10, 20]}
grid_rf = GridSearchCV(RandomForestClassifier(class_weight='balanced', random_state=42), param_grid_rf, cv=3, scoring='f1')
grid_rf.fit(X_train, y_train)
y_pred_grid_rf = grid_rf.predict(X_test)
print(f'Best Params: {grid_rf.best_params_}')
print(classification_report(y_test, y_pred_grid_rf))

##### Which hyperparameter optimization technique have you used and why?

GridSearchCV.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Random Forest provides robust baseline without tuning.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

Precision avoids false positives. Recall finds all negatives. F1-score balances both.

### ML Model - 3

In [ ]:
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
print('XGBoost Performance:')
print(classification_report(y_test, y_pred_xgb))
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges')
plt.title('XGBoost Confusion Matrix')
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Grid search skipped for brevity on XGBoost as it's computationally heavy.
print('Skipped due to computation time.')

##### Which hyperparameter optimization technique have you used and why?

None for XGBoost.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

XGBoost often outperforms others even without heavy tuning.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

F1-Score, as it balances precision and recall, especially important for imbalanced sentiment datasets.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

Logistic Regression (with TF-IDF). It is highly interpretable, fast, and often achieves near-state-of-the-art performance on simple text classification tasks.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

Logistic Regression coefficients indicate feature importance. Words with highest positive coefficients drive positive sentiment.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
joblib.dump(grid_lr.best_estimator_, 'zomato_sentiment_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
loaded_model = joblib.load('zomato_sentiment_model.pkl')
loaded_tfidf = joblib.load('tfidf_vectorizer.pkl')
sample_text = ['The food was terrible and cold', 'Absolutely loved the ambiance and the taste']
sample_vec = loaded_tfidf.transform(sample_text)
print('Predictions:', loaded_model.predict(sample_vec))

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

We successfully built a Sentiment Analysis model to classify Zomato reviews. Through EDA, we discovered that while cost and cuisine play roles in popularity, the text of the review is the strongest indicator of sentiment. The Logistic Regression model paired with TF-IDF provides a robust, interpretable, and deployable solution for automatically tagging review sentiments.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***